# Localization Sample : Kalman filter (EKF) with constant velocity 

---- 

- conda env : [ai_robotics](../../README.md#setup-a-conda-environment)

---

### Ref
- https://github.com/AtsushiSakai/PythonRobotics/


| Property           | [EKF](./extended_kalman_filter.ipynb) | [KF](./kalman_filter_constant_velocity.ipynb)                             |
| ------------------ | ------------------------------- | ------------------------------ |
| Motion model       | Nonlinear                       | Linear                         |
| Requires Jacobians | ✅ Yes                           | ❌ No                           |
| State example      | [x, y, yaw, v]                  | [x, y, vx, vy]                 |
| Best for           | Robot localization, turn motion | Linear systems, radar tracking |
| Computation        | Higher                          | Faster, simpler                |


### Imports and Configuration

In [1]:
import sys
from pathlib import Path

parent_dir = str(Path().resolve().parents[0])
# Add this path to sys.path
sys.path.insert(0, parent_dir)


In [2]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from IPython.display import HTML
from utils.plot import plot_covariance_ellipse

# Simulation parameters
DT = 0.1          # time step [s]
SIM_TIME = 20.0   # total simulation time [s]

# Process noise covariance (model uncertainty)
Q = np.diag([0.1, 0.1, 0.5, 0.5]) ** 2

# Observation noise covariance (measurement noise)
R = np.diag([0.5, 0.5]) ** 2


### Motion and Functions

In [3]:
def motion_model(x):
    """
    Constant velocity motion model.
    x_k+1 = F x_k + w
    """
    F = np.array([
        [1, 0, DT, 0],
        [0, 1, 0, DT],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ])
    return F @ x, F

def observation_model(x):
    """
    Measure only position (x, y).
    z_k = H x_k + v
    """
    H = np.array([
        [1, 0, 0, 0],
        [0, 1, 0, 0]
    ])
    z = H @ x
    return z, H

def simulate_motion(xTrue):
    """Simulate true motion."""
    F = np.array([
        [1, 0, DT, 0],
        [0, 1, 0, DT],
        [0, 0, 1, 0],
        [0, 0, 0, 1]
    ])
    return F @ xTrue

def simulate_observation(xTrue):
    """Simulate noisy position observation."""
    z, _ = observation_model(xTrue)
    noise = R @ np.random.randn(2, 1)
    return z + noise

def kalman_filter(xEst, PEst, z):
    """Standard linear Kalman filter equations."""
    # Predict
    xPred, F = motion_model(xEst)
    PPred = F @ PEst @ F.T + Q

    # Update
    zPred, H = observation_model(xPred)
    y = z - zPred
    S = H @ PPred @ H.T + R
    K = PPred @ H.T @ np.linalg.inv(S)
    xEst = xPred + K @ y
    PEst = (np.eye(len(xEst)) - K @ H) @ PPred

    return xEst, PEst



### Animation Setup and Update Function

In [5]:
def run_kf_animation():
    # Initialize true state, estimate, covariance
    xTrue = np.array([[0.0], [0.0], [1.0], [0.5]])  # starting at origin
    xEst = np.zeros((4, 1))
    PEst = np.eye(4)
    time = 0.0

    hxTrue = xTrue
    hxEst = xEst
    hz = np.zeros((2, 1))

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.set_aspect('equal')
    ax.grid(True)

    def init():
        ax.cla()
        ax.grid(True)
        ax.set_aspect('equal')
        return []

    def update(frame):
        nonlocal xTrue, xEst, PEst, hxTrue, hxEst, hz, time
        time += DT
        if time > SIM_TIME:
            ani.event_source.stop()
            return []

        # Simulate true motion and noisy observation
        xTrue = simulate_motion(xTrue)
        z = simulate_observation(xTrue)

        # Kalman filter update
        xEst, PEst = kalman_filter(xEst, PEst, z)

        # Record history
        hxTrue = np.hstack((hxTrue, xTrue))
        hxEst = np.hstack((hxEst, xEst))
        hz = np.hstack((hz, z))

        # Clear previous frame
        ax.cla()
        ax.grid(True)
        ax.set_aspect('equal')
        # ax.set_aspect('equal', adjustable='datalim')

        # Auto-scale the axes
        all_x = np.hstack((hxTrue[0, :], hxEst[0, :]))
        all_y = np.hstack((hxTrue[1, :], hxEst[1, :]))
        x_min, x_max = np.min(all_x), np.max(all_x)
        y_min, y_max = np.min(all_y), np.max(all_y)
        margin_x = (x_max - x_min) * 0.2 + 0.5
        margin_y = (y_max - y_min) * 0.2 + 0.5
        ax.set_xlim(x_min - margin_x, x_max + margin_x)
        ax.set_ylim(y_min - margin_y, y_max + margin_y)

        # Draw trajectories
        ax.plot(hz[0, :], hz[1, :], ".g", label="observation")
        ax.plot(hxTrue[0, :], hxTrue[1, :], "-b", label="true")
        ax.plot(hxEst[0, :], hxEst[1, :], "-r", label="KF estimate")
        plot_covariance_ellipse(xEst[0, 0], xEst[1, 0], PEst[:2, :2], ax=ax)

        ax.legend(loc='lower right', fontsize='small', frameon=True, shadow=True)
        ax.set_title(f"Kalman Filter (Constant Velocity) | Time = {time:.1f}s")

        return []

    ani = FuncAnimation(
        fig,
        update,
        frames=int(SIM_TIME / DT),
        init_func=init,
        interval=100,
        blit=False,
        repeat=False
    )
    plt.close(fig)  # prevent duplicate static plot
    return ani

SIM_TIME = 3 # Update the duration of total simulation time [s]
ani = run_kf_animation()
HTML(ani.to_html5_video())
